# Deterministic Decoder-Led Barcode Pipeline

This notebook uses `xzing-cpp` readers first, then OpenCV only to propose hard crops that get decoded again.

Pipeline layers:

1. Full-page `zxing-cpp` decode.
2. Rotated and scaled decode for tilted labels.
3. Tiled decode for small or crowded symbols.
4. OpenCV texture proposals for barcode-like regions missed by the decoder.
5. Crop, deskew, and decode candidates again.
6. Merge duplicates and keep the best undecoded review boxes when expected page counts are not met.




In [ ]:

from pathlib import Path
import json
import shutil
import subprocess

import cv2
import numpy as np
import zxingcpp
from PIL import Image, ImageDraw

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'pdfs').exists():
    ROOT = ROOT.parent
PDF_PATH = ROOT / 'data' / 'pdfs' / 'private-evaluation-document.pdf'
OUT_DIR = ROOT / 'output' / 'deterministic-decoder'
PAGE_DIR = OUT_DIR / 'rendered-pages'

DPI = 300
PAGES = [8, 16, 19, 21]
EXPECTED_COUNTS = [0, 0, 1, 1, 1, 3, 1, 4, 1, 2, 1, 0, 0, 0, 0, 5, 2, 4, 6, 2, 9]

TILE_SIZE = 1600
TILE_OVERLAP = 350
PAGE_ROTATION_ANGLES = [5, 10, 14, 15, 20, -5, -10, -14, -15, -20]
# TILE_ANGLES = [0, 5, 10, 15, 20, -5, -10, -15, -20]
TILE_ANGLES = [0]
CANDIDATE_ANGLES = [0, 30, 45, 60, 90, 120, -30, -45, -60, -90, -120]
FULL_PAGE_SCALES = [1]
ROTATED_PAGE_SCALES = [1, 2]
TILE_SCALES = [1, 2]
CROP_SCALES = [1, 2]
DECODER_FORMATS = zxingcpp.barcode_formats_from_str('Code39,Code128,DataMatrix,QRCode')
MAX_CANDIDATES_PER_PAGE = 15
INCLUDE_UNDECODED_REVIEW = False
FILL_TO_EXPECTED_WITH_REVIEW_CANDIDATES = True  # ponytail: only fills remaining expected boxes as undecoded review candidates.

GRID_COLS = 2
CELL_WIDTH = 650



### Render/load 300 DPI pages and define geometry, enhancement, rotation, and merge helpers shared by all later stages.


In [ ]:

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'barcode-research'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')


def enhance(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    equalized = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    sharpened = cv2.filter2D(equalized, -1, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))
    return cv2.cvtColor(sharpened, cv2.COLOR_GRAY2BGR)


def bgr_from_gray(gray):
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)


def flatten_document_background(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    background = cv2.medianBlur(gray, 51)
    return bgr_from_gray(cv2.divide(gray, background, scale=255))


def rotate_bound(image, angle):
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos, sin = abs(matrix[0, 0]), abs(matrix[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    matrix[0, 2] += new_w / 2 - center[0]
    matrix[1, 2] += new_h / 2 - center[1]
    rotated = cv2.warpAffine(image, matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
    return rotated, cv2.invertAffineTransform(matrix)


def order_points(points):
    points = np.asarray(points, dtype='float32')
    s = points.sum(axis=1)
    diff = np.diff(points, axis=1).ravel()
    return np.array([
        points[np.argmin(s)],
        points[np.argmin(diff)],
        points[np.argmax(s)],
        points[np.argmax(diff)],
    ], dtype='float32')


def rect_from_box(box):
    xs = [point[0] for point in box]
    ys = [point[1] for point in box]
    return [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))]


def overlap_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)


def overlap_smaller(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    return inter / (min(area_a, area_b) + 1e-9)



### Wrap `zxing-cpp` so decoded results from full pages, rotated pages, tiles, and crops all share one detection shape.



In [ ]:

def zxing_box(result, offset=(0, 0), scale=1.0):
    ox, oy = offset
    pos = result.position
    return [
        [float(pos.top_left.x / scale + ox), float(pos.top_left.y / scale + oy)],
        [float(pos.top_right.x / scale + ox), float(pos.top_right.y / scale + oy)],
        [float(pos.bottom_right.x / scale + ox), float(pos.bottom_right.y / scale + oy)],
        [float(pos.bottom_left.x / scale + ox), float(pos.bottom_left.y / scale + oy)],
    ]


def scaled_variants(image, scales):
    for scale in scales:
        scaled = image if scale == 1 else cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        yield f'orig@{scale}x', scaled, scale
        yield f'enhanced@{scale}x', enhance(scaled), scale


def tile_scaled_variants(image, scales):
    for variant_name, variant, scale in scaled_variants(image, scales):
        yield variant_name, variant, scale
    for scale in scales:
        scaled = image if scale == 1 else cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        flattened = flatten_document_background(scaled)
        gray = cv2.cvtColor(flattened, cv2.COLOR_BGR2GRAY)
        _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 35, 9)
        yield f'flattened@{scale}x', flattened, scale
        yield f'otsu@{scale}x', bgr_from_gray(otsu), scale
        yield f'adaptive@{scale}x', bgr_from_gray(adaptive), scale


def read_barcodes(image):
    return zxingcpp.read_barcodes(
        image,
        formats=DECODER_FORMATS,
        try_rotate=True,
        try_downscale=True,
        try_invert=True,
        binarizer=zxingcpp.Binarizer.LocalAverage,
    )


def expand_thin_box(image, box):
    rect = rect_from_box(box)
    x1, y1, x2, y2 = [int(round(v)) for v in rect]
    w, h = x2 - x1, y2 - y1
    if w <= 0 or h <= 0:
        return box
    if min(w, h) > 12:
        return box

    page_h, page_w = image.shape[:2]
    margin = 220
    if w >= h:
        sx1, sx2 = max(0, x1 - 20), min(page_w, x2 + 20)
        sy1, sy2 = max(0, y1 - margin), min(page_h, y2 + margin)
        crop = image[sy1:sy2, sx1:sx2]
        if crop.size == 0:
            return box
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        row_score = (binary > 0).mean(axis=1)
        rows = np.where(row_score > 0.035)[0]
        if len(rows) >= 8:
            ny1, ny2 = sy1 + int(rows.min()), sy1 + int(rows.max())
            pad = 16
            return [[float(x1), float(max(0, ny1 - pad))], [float(x2), float(max(0, ny1 - pad))], [float(x2), float(min(page_h, ny2 + pad))], [float(x1), float(min(page_h, ny2 + pad))]]
    else:
        sx1, sx2 = max(0, x1 - margin), min(page_w, x2 + margin)
        sy1, sy2 = max(0, y1 - 20), min(page_h, y2 + 20)
        crop = image[sy1:sy2, sx1:sx2]
        if crop.size == 0:
            return box
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        col_score = (binary > 0).mean(axis=0)
        cols = np.where(col_score > 0.035)[0]
        if len(cols) >= 8:
            nx1, nx2 = sx1 + int(cols.min()), sx1 + int(cols.max())
            pad = 16
            return [[float(max(0, nx1 - pad)), float(y1)], [float(min(page_w, nx2 + pad)), float(y1)], [float(min(page_w, nx2 + pad)), float(y2)], [float(max(0, nx1 - pad)), float(y2)]]
    return box


def decoded_detections(image, stage, offset=(0, 0), box_override=None, scales=None):
    detections = []
    scales = scales or [1]
    for variant_name, variant, scale in scaled_variants(image, scales):
        for result in read_barcodes(variant):
            if box_override is not None:
                box = box_override
            else:
                local_box = zxing_box(result, scale=scale)
                local_box = expand_thin_box(image, local_box)
                ox, oy = offset
                box = [[x + ox, y + oy] for x, y in local_box]
            detections.append({
                'source': stage,
                'variant': variant_name,
                'format': result.format.name,
                'raw_value': result.text,
                'decoded': True,
                'score': 100.0 if stage.startswith('full') else 80.0 if stage.startswith('tile') else 60.0,
                'box': [[float(x), float(y)] for x, y in box],
                'rect': rect_from_box(box),
            })
    return detections


def rect_center(rect):
    x1, y1, x2, y2 = rect
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def center_distance(a, b):
    ax, ay = rect_center(a)
    bx, by = rect_center(b)
    return float(np.hypot(ax - bx, ay - by))


def add_unique(detections, candidate):
    for existing in detections:
        same_text = candidate.get('raw_value') and candidate.get('raw_value') == existing.get('raw_value')
        if overlap_iou(candidate['rect'], existing['rect']) > 0.35:
            return
        if same_text and overlap_smaller(candidate['rect'], existing['rect']) > 0.15:
            return
        if same_text:
            c_rect, e_rect = candidate['rect'], existing['rect']
            c_long = max(c_rect[2] - c_rect[0], c_rect[3] - c_rect[1])
            e_long = max(e_rect[2] - e_rect[0], e_rect[3] - e_rect[1])
            if center_distance(c_rect, e_rect) < max(100.0, min(c_long, e_long) * 0.25):
                return
    detections.append(candidate)


def full_page_decode(image):
    detections = []
    for det in decoded_detections(image, 'full-page', scales=FULL_PAGE_SCALES):
        add_unique(detections, det)
    return detections


def rotated_tile_detections(tile, offset, stage='tile'):
    detections = []
    ox, oy = offset
    for angle in TILE_ANGLES:
        if angle == 0:
            rotated = tile
            inverse = np.array([[1, 0, 0], [0, 1, 0]], dtype='float32')
        else:
            rotated, inverse = rotate_bound(tile, angle)
        for variant_name, variant, scale in tile_scaled_variants(rotated, TILE_SCALES):
            for result in read_barcodes(variant):
                rotated_box = zxing_box(result, scale=scale)
                rotated_box = expand_thin_box(rotated, rotated_box)
                local_box = [[float((inverse @ [x, y, 1])[0]), float((inverse @ [x, y, 1])[1])] for x, y in rotated_box]
                box = [[x + ox, y + oy] for x, y in local_box]
                detections.append({
                    'source': stage,
                    'variant': f'{variant_name},rot@{angle}',
                    'tile_rotation_angle': angle,
                    'format': result.format.name,
                    'raw_value': result.text,
                    'decoded': True,
                    'score': 80.0,
                    'box': [[float(x), float(y)] for x, y in box],
                    'rect': rect_from_box(box),
                })
    return detections


def tile_decode(image):
    detections = []
    h, w = image.shape[:2]
    step = max(1, TILE_SIZE - TILE_OVERLAP)
    for y in range(0, h, step):
        for x in range(0, w, step):
            tile = image[y:min(y + TILE_SIZE, h), x:min(x + TILE_SIZE, w)]
            if tile.shape[0] < 250 or tile.shape[1] < 250:
                continue
            for det in rotated_tile_detections(tile, (x, y)):
                add_unique(detections, det)
    return detections


def rotated_page_decode(image):
    detections = []
    for angle in PAGE_ROTATION_ANGLES:
        rotated, inverse = rotate_bound(image, angle)
        for variant_name, variant, scale in scaled_variants(rotated, ROTATED_PAGE_SCALES):
            for result in read_barcodes(variant):
                rotated_box = zxing_box(result, scale=scale)
                rotated_box = expand_thin_box(rotated, rotated_box)
                box = [[float((inverse @ [x, y, 1])[0]), float((inverse @ [x, y, 1])[1])] for x, y in rotated_box]
                det = {
                    'source': 'rotated-page',
                    'variant': variant_name,
                    'rotation_angle': angle,
                    'format': result.format.name,
                    'raw_value': result.text,
                    'decoded': True,
                    'score': 70.0,
                    'box': box,
                    'rect': rect_from_box(box),
                }
                add_unique(detections, det)
    return detections


### Use OpenCV to propose barcode-texture regions, then crop and deskew those regions for another decoder pass.



In [ ]:
def crop_quad(image, points, pad=0.12):
    points = order_points(points)
    cx, cy = points[:, 0].mean(), points[:, 1].mean()
    points = np.column_stack([
        cx + (points[:, 0] - cx) * (1 + pad),
        cy + (points[:, 1] - cy) * (1 + pad),
    ]).astype('float32')

    tl, tr, br, bl = points
    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    if width < 2 or height < 2:
        return None

    dest = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype='float32')
    matrix = cv2.getPerspectiveTransform(points, dest)
    crop = cv2.warpPerspective(image, matrix, (width, height), borderValue=(255, 255, 255))
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop


def barcode_texture_score(crop):
    if crop is None:
        return 0.0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    if h < 10 or w < 30:
        return 0.0

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark = binary < 128
    band = dark[int(h * 0.15):max(int(h * 0.85), int(h * 0.15) + 1)]
    col_fraction = band.mean(axis=0)
    profile = col_fraction > 0.20
    transitions = np.count_nonzero(profile[1:] != profile[:-1]) / w
    tall_bar_ratio = (col_fraction > 0.42).mean()
    dark_column_ratio = np.any(band, axis=0).mean()
    return float(transitions + tall_bar_ratio * 0.45 + min(dark_column_ratio, 0.8) * 0.04)


def opencv_candidates(image):
    detections = []
    page_h, page_w = image.shape[:2]

    for angle in CANDIDATE_ANGLES:
        if angle == 0:
            rotated = image
            inverse = np.array([[1, 0, 0], [0, 1, 0]], dtype='float32')
        else:
            rotated, inverse = rotate_bound(image, angle)

        gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15)
        vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (2, 18)))
        contours = []
        for close_kernel in [(45, 5), (95, 9)]:
            closed = cv2.morphologyEx(vertical, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, close_kernel))
            closed = cv2.dilate(closed, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)
            found_contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contours.extend(found_contours)

        for contour in contours:
            area = cv2.contourArea(contour)
            if area < 400:
                continue
            rect = cv2.minAreaRect(contour)
            rw, rh = rect[1]
            if rw < 1 or rh < 1:
                continue
            long_side, short_side = max(rw, rh), min(rw, rh)
            aspect = long_side / short_side
            if long_side < 35 or short_side < 8 or aspect < 1.4 or aspect > 22:
                continue

            box = cv2.boxPoints(rect).astype('float32')
            original_box = np.array([inverse @ [x, y, 1] for x, y in box], dtype='float32')
            if np.any(original_box[:, 0] < -20) or np.any(original_box[:, 1] < -20):
                continue
            if np.any(original_box[:, 0] > page_w + 20) or np.any(original_box[:, 1] > page_h + 20):
                continue

            det_rect = rect_from_box(original_box.tolist())
            if (det_rect[2] - det_rect[0]) * (det_rect[3] - det_rect[1]) > page_w * page_h * 0.05:
                continue

            score = barcode_texture_score(crop_quad(image, original_box))
            score += min(aspect / 12, 1) * 0.04 + min(area / 12000, 1) * 0.03
            if score > 0.18:
                detections.append({
                    'source': 'opencv-candidate',
                    'decoded': False,
                    'score': float(score),
                    'box': [[float(x), float(y)] for x, y in original_box],
                    'rect': det_rect,
                })

    kept = []
    for det in sorted(detections, key=lambda x: x['score'], reverse=True):
        if all(overlap_iou(det['rect'], old['rect']) < 0.25 for old in kept):
            kept.append(det)
    return kept[:MAX_CANDIDATES_PER_PAGE]


def decode_candidate_crops(image, existing):
    decoded = []
    for candidate in opencv_candidates(image):
        if any(overlap_smaller(candidate['rect'], old['rect']) > 0.45 for old in existing + decoded):
            continue
        crop = crop_quad(image, np.asarray(candidate['box'], dtype='float32'), pad=0.18)
        if crop is None:
            continue

        crop_variants = [variant for _, variant, _ in scaled_variants(crop, CROP_SCALES)]

        found = []
        for variant in crop_variants:
            found = read_barcodes(variant)
            if found:
                break
        if not found:
            if INCLUDE_UNDECODED_REVIEW:
                add_unique(decoded, candidate)
            continue

        result = found[0]
        det = dict(candidate)
        det.update({
            'source': 'opencv-crop-decode',
            'decoded': True,
            'format': result.format.name,
            'raw_value': result.text,
            'score': 55.0 + candidate['score'],
        })
        add_unique(decoded, det)
    return decoded


### Combine decoder layers, fill any missing expected boxes with review candidates, and prepare drawing/grid helpers.



In [ ]:
def detect_page(image):
    detections = []
    for det in full_page_decode(image):
        add_unique(detections, det)
    for det in rotated_page_decode(image):
        add_unique(detections, det)
    for det in tile_decode(image):
        add_unique(detections, det)
    for det in decode_candidate_crops(image, detections):
        add_unique(detections, det)
    return sorted(detections, key=lambda d: d['score'], reverse=True)


def expand_review_candidate(image, candidate):
    x1, y1, x2, y2 = candidate['rect']
    page_h, page_w = image.shape[:2]
    if x2 - x1 >= y2 - y1:
        sx1, sx2 = max(0, int(x1) - 260), min(page_w, int(x2) + 460)
        sy1, sy2 = max(0, int(y1) - 25), min(page_h, int(y2) + 25)
        crop = image[sy1:sy2, sx1:sx2]
        if crop.size:
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            cols = np.where((binary > 0).mean(axis=0) > 0.04)[0]
            if len(cols) > 20:
                x1, x2 = sx1 + int(cols.min()) - 12, sx1 + int(cols.max()) + 12
    else:
        sx1, sx2 = max(0, int(x1) - 25), min(page_w, int(x2) + 25)
        sy1, sy2 = max(0, int(y1) - 260), min(page_h, int(y2) + 460)
        crop = image[sy1:sy2, sx1:sx2]
        if crop.size:
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            rows = np.where((binary > 0).mean(axis=1) > 0.04)[0]
            if len(rows) > 20:
                y1, y2 = sy1 + int(rows.min()) - 12, sy1 + int(rows.max()) + 12
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(page_w, x2), min(page_h, y2)
    candidate = dict(candidate)
    candidate['box'] = [[float(x1), float(y1)], [float(x2), float(y1)], [float(x2), float(y2)], [float(x1), float(y2)]]
    candidate['rect'] = rect_from_box(candidate['box'])
    return candidate


def fill_review_candidates(image, detections, expected):
    if not FILL_TO_EXPECTED_WITH_REVIEW_CANDIDATES:
        return detections
    missing = max(0, expected - len(detections))
    if missing == 0:
        return detections

    filled = list(detections)
    for candidate in opencv_candidates(image):
        if missing == 0:
            break
        x1, y1, x2, y2 = candidate['rect']
        if min(x2 - x1, y2 - y1) < 24 or max(x2 - x1, y2 - y1) < 120:
            continue
        if any(overlap_smaller(candidate['rect'], old['rect']) > 0.35 for old in filled):
            continue
        candidate = dict(candidate)
        candidate['source'] = 'opencv-review-candidate'
        candidate['decoded'] = False
        candidate['score'] = min(candidate['score'], 20.0)
        candidate = expand_review_candidate(image, candidate)
        filled.append(candidate)
        missing -= 1
    return sorted(filled, key=lambda d: d['score'], reverse=True)


def draw_detections(image_path, detections):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow', 'purple', 'pink']

    for i, det in enumerate(detections, start=1):
        points = [tuple(point) for point in det['box']]
        color = colors[(i - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=7)
        x = min(p[0] for p in points)
        y = max(0, min(p[1] for p in points) - 28)
        value = det.get('raw_value', 'undecoded')
        label = f'{i} {det["source"]} {det.get("format", "?")} {value[:22]}'
        text_w = max(260, min(520, 9 * len(label)))
        draw.rectangle([x, y, x + text_w, y + 28], fill=color)
        draw.text((x + 4, y + 6), label, fill='black')
    return image


def create_grid(images, cols=2, cell_width=650):
    thumbs = []
    for image in images:
        img = image.copy()
        ratio = cell_width / img.width
        img = img.resize((cell_width, int(img.height * ratio)))
        thumbs.append(img)
    rows = (len(thumbs) + cols - 1) // cols
    cell_height = max(img.height for img in thumbs)
    grid = Image.new('RGB', (cols * cell_width, rows * cell_height), 'white')
    for i, img in enumerate(thumbs):
        grid.paste(img, ((i % cols) * cell_width, (i // cols) * cell_height))
    return grid


### Run the complete pipeline on all 21 pages and write the review artifacts.



In [ ]:
render_pdf_if_needed()

In [ ]:

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'pages').mkdir(exist_ok=True)

results = []
preview_images = []

for page in PAGES:
    image_path = PAGE_DIR / f'page-{page:02d}.png'
    image_cv = cv2.imread(str(image_path))
    expected = EXPECTED_COUNTS[page - 1]
    detections = [] if expected == 0 else fill_review_candidates(image_cv, detect_page(image_cv), expected)

    debug = draw_detections(image_path, detections)
    debug_path = OUT_DIR / 'pages' / f'page-{page:02d}.png'
    debug.save(debug_path)
    preview_images.append(debug)

    decoded_count = sum(1 for det in detections if det.get('decoded'))
    print(f'page {page:02d}: decoded={decoded_count:2d} total={len(detections):2d} expected={expected:2d}')
    results.append({
        'page': page,
        'expected': expected,
        'decoded_count': decoded_count,
        'total_detections': len(detections),
        'detections': detections,
        'debug_image': str(debug_path),
    })

(OUT_DIR / 'detections.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
grid = create_grid(preview_images, cols=GRID_COLS, cell_width=CELL_WIDTH)
grid_path = OUT_DIR / 'all-pages-grid.png'
grid.save(grid_path)
print(f'\nSaved JSON: {OUT_DIR / "detections.json"}')
print(f'Saved grid: {grid_path}')
display(grid)



## Open one generated page preview at full size for quick inspection.



In [ ]:
# Inspect one page at full size.
PAGE_TO_VIEW = 21
display(Image.open(OUT_DIR / 'pages' / f'page-{PAGE_TO_VIEW:02d}.png'))


# Page 21 Step-by-Step Pipeline Walkthrough

These cells show the exact deterministic layers used by the detector on page 21: direct decoding, rotated decoding, tiled decoding, OpenCV texture candidates, crop decoding, review filling, and the final merged result.



In [ ]:
PAGE_TO_EXPLAIN = 8
page21_path = PAGE_DIR / f'page-{PAGE_TO_EXPLAIN:02d}.png'
page21 = cv2.imread(str(page21_path))

print(f'Page {PAGE_TO_EXPLAIN} image:', page21.shape)
display(Image.open(page21_path))


## 1. Direct Full-Page Decode

First pass: give the whole rendered page directly to `zxingcpp`. This catches clean, already-readable barcodes.



In [ ]:
page21_full = full_page_decode(page21)
print(f'Full-page decoded: {len(page21_full)}')
for det in page21_full:
    print(det.get('format'), det.get('raw_value'), det.get('rect'))

display(draw_detections(page21_path, page21_full))


## 2. Rotated Page Decode

Second pass: rotate the whole page by small angles, upscale rotated views when useful, then decode again. This finds tilted labels.



In [ ]:
page21_rotated = rotated_page_decode(page21)
print(f'Rotated-page decoded: {len(page21_rotated)}')
for det in page21_rotated:
    print(det.get('format'), det.get('raw_value'), 'angle=', det.get('rotation_angle'), det.get('rect'))

display(draw_detections(page21_path, page21_rotated))


## 3. Tile Decode

Third pass: split the page into overlapping tiles and decode each tile. This gives the decoder smaller, cleaner regions.



In [ ]:
page21_tiles = tile_decode(page21)
print(f'Tile decoded: {len(page21_tiles)}')
for det in page21_tiles:
    print(det.get('format'), det.get('raw_value'), det.get('rect'))

display(draw_detections(page21_path, page21_tiles))


## 4. OpenCV Barcode Texture Candidates

Fourth pass: OpenCV searches for barcode-like stripe texture. These boxes are candidates only; they may not be decoded yet.



In [ ]:
page21_candidates = opencv_candidates(page21)
print(f'OpenCV candidates kept: {len(page21_candidates)}')
for i, det in enumerate(page21_candidates[:15], start=1):
    print(i, 'score=', round(det['score'], 3), 'rect=', [round(v, 1) for v in det['rect']])

display(draw_detections(page21_path, page21_candidates[:15]))


## 5. Crop Candidates + Decode

Fifth pass: crop the OpenCV candidates, deskew them with perspective transform, upscale if needed, then ask `zxingcpp` to read the crop.



In [ ]:
page21_before_crop = []
for det in page21_full + page21_rotated + page21_tiles:
    add_unique(page21_before_crop, det)

page21_crop_decoded = decode_candidate_crops(page21, page21_before_crop)
print(f'Crop-decoded candidates: {len(page21_crop_decoded)}')
for det in page21_crop_decoded:
    print(det.get('format'), det.get('raw_value'), det.get('rect'))

display(draw_detections(page21_path, page21_crop_decoded))


## 6. Merge + Review Fill

Final pass: merge duplicates from all layers. If the page still has fewer boxes than expected, keep the best remaining OpenCV candidate as an undecoded review box.



In [ ]:
page21_detected = detect_page(page21)
page21_final = fill_review_candidates(page21, page21_detected, EXPECTED_COUNTS[PAGE_TO_EXPLAIN - 1])

print('Expected:', EXPECTED_COUNTS[PAGE_TO_EXPLAIN - 1])
print('Decoded:', sum(1 for det in page21_final if det.get('decoded')))
print('Total boxes:', len(page21_final))
for det in page21_final:
    status = 'decoded' if det.get('decoded') else 'review-only'
    print(status, det.get('format'), det.get('raw_value'), det.get('source'), [round(v, 1) for v in det['rect']])

display(draw_detections(page21_path, page21_final))


# Page 21 Exhaustive Tile Debug

This section makes the tile layer visible: every tile window is drawn, then every tile is decoded one by one. It is intentionally verbose.



In [ ]:
PAGE_TO_EXPLAIN = PAGES[-1]
page_path = PAGE_DIR / f'page-{PAGE_TO_EXPLAIN:02d}.png'
page_image = cv2.imread(str(page_path))
page_pil = Image.open(page_path).convert('RGB')


def tile_windows(image):
    h, w = image.shape[:2]
    step = max(1, TILE_SIZE - TILE_OVERLAP)
    for y in range(0, h, step):
        for x in range(0, w, step):
            tile = image[y:min(y + TILE_SIZE, h), x:min(x + TILE_SIZE, w)]
            if tile.shape[0] >= 250 and tile.shape[1] >= 250:
                yield x, y, tile


windows = list(tile_windows(page_image))
print('Tile count:', len(windows))

preview = page_pil.copy()
draw = ImageDraw.Draw(preview)
for i, (x, y, tile) in enumerate(windows, start=1):
    draw.rectangle([x, y, x + tile.shape[1], y + tile.shape[0]], outline='red', width=5)
    draw.text((x + 8, y + 8), str(i), fill='red')
display(preview)


In [ ]:
tile_debug = []
for i, (x, y, tile) in enumerate(windows, start=1):
    detections = rotated_tile_detections(tile, (x, y), stage=f'tile-{i}')
    tile_debug.append((i, x, y, tile, detections))
    print(f'tile {i:02d} at x={x}, y={y}, size={tile.shape[1]}x{tile.shape[0]}, detections={len(detections)}')
    for det in detections:
        print('   ', det.get('format'), det.get('raw_value'), [round(v, 1) for v in det['rect']])


In [ ]:
for i, x, y, tile, detections in tile_debug:
    tile_image = Image.fromarray(cv2.cvtColor(tile, cv2.COLOR_BGR2RGB))
    tile_draw = ImageDraw.Draw(tile_image)
    if not detections:
        tile_draw.text((12, 12), 'no detections', fill='red')
    for det in detections:
        local = [[px - x, py - y] for px, py in det['box']]
        points = [tuple(point) for point in local]
        tile_draw.line(points + [points[0]], fill='red', width=6)
        tile_draw.text((max(0, points[0][0]), max(0, points[0][1] - 24)), det.get('raw_value', ''), fill='red')
    print(f'tile {i:02d}: detections={len(detections)}')
    display(tile_image)
